# D3 DP-aware MIA across epsilon

Experiment 27 at epsilon=3 reported MI AUC = 0.891, statistically indistinguishable from the no-defense baseline of 0.878. The natural follow-up: at what epsilon does the DP-aware MIA actually drop? The Yeom (2018) bound predicts epsilon=1 should bring MI advantage to ~0.63, and epsilon=0.5 to ~0.39. This notebook tests both.

Two passes, each with 8 DP-trained shadows + 1 DP-trained target at the named epsilon:

    epsilon = 1.0  (Yeom upper bound on MI advantage ~ 0.63)
    epsilon = 0.5  (Yeom upper bound on MI advantage ~ 0.39)

Output JSONs: results/27_d3_membership_aware_attacker_eps1.0.json and results/27_d3_membership_aware_attacker_eps0.5.json.

**Runtime -> A100 strongly recommended.** Expected wall: ~5-6 h on A100 (~9 DP-SGD trainings per epsilon point x 2 points). On L4 this is 10-12 h, split across sessions.

**Don't `Save a copy in GitHub` from Colab.**

In [ ]:
!rm -rf /content/bci-identity-leakage
!git clone https://github.com/manrajmondair/bci-identity-leakage.git /content/bci-identity-leakage
%cd /content/bci-identity-leakage
!git rev-parse HEAD
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
import torch
tv = torch.__version__.split('+')[0]
!pip install --quiet "torchaudio=={tv}"
!pip install --quiet mne moabb pyriemann braindecode opacus httpx tqdm rich scipy

In [ ]:
!python -m data.prefetch_direct --runs imagery --workers 8

## Pass 1: epsilon = 1.0

In [ ]:
!PYTHONUNBUFFERED=1 python -u -m experiments.27_d3_membership_aware_attacker \
  --all --target-epsilon 1.0

## Pass 2: epsilon = 0.5

In [ ]:
!PYTHONUNBUFFERED=1 python -u -m experiments.27_d3_membership_aware_attacker \
  --all --target-epsilon 0.5

In [ ]:
import json, os, subprocess, datetime, platform, sys
import torch
if os.path.isdir('/content/bci-identity-leakage'): os.chdir('/content/bci-identity-leakage')
TAG = 'd3_dp_aware_mia_sweep'
try:
    git_sha = subprocess.check_output(['git','rev-parse','HEAD']).decode().strip()
except Exception: git_sha = 'unknown'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
now_utc = datetime.datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
run_id = now_utc.replace(':','').replace('-','').rstrip('Z') + f'_{TAG}_{git_sha[:7]}'
outputs = ['results/27_d3_membership_aware_attacker_eps1.0.json',
           'results/27_d3_membership_aware_attacker_eps0.5.json']
meta = {'run_id': run_id, 'experiment': 'experiments.27_d3_membership_aware_attacker',
        'git_sha': git_sha, 'hardware': f'Colab {gpu}',
        'platform': platform.platform(), 'torch_version': torch.__version__,
        'completed_at_utc': now_utc, 'outputs': outputs}
os.makedirs(f'runs/{run_id}', exist_ok=True)
with open(f'runs/{run_id}/meta.json','w') as f: json.dump(meta, f, indent=2)
for path in outputs:
    print(f'--- BEGIN {os.path.basename(path)} ---')
    if os.path.exists(path):
        print(open(path).read())
    else:
        print(f'MISSING: {path}')
    print(f'--- END {os.path.basename(path)} ---')
    print()
print('--- BEGIN run meta ---')
print(json.dumps(meta, indent=2))
print('--- END run meta ---')